# Unión completa — train_2016_2017 + stores + transactions + items + holidays_events + oil

Parte de `train_2016_2017.csv` (ya filtrado a 2016-2017, todas las tiendas) y le une, todo por *left join* (para no perder ninguna fila de `train`):

1. `stores.csv` por `store_nbr`
2. `transactions.csv` por `date` + `store_nbr`
3. `items.csv` por `item_nbr`
4. `holidays_events.csv` por `date` (deduplicado primero)
5. `oil.csv` por `date`

`test.csv` y `sample_submission.csv` quedan fuera (no comparten `id` con `train`, no tiene sentido unirlas así).

**Antes de correr:** asegúrate de haber generado antes `train_2016_2017.csv` (notebook `bbdd.ipynb`).

In [ ]:
import pandas as pd
import os

BASE_DIR = "C:/Tesis"


## 1. Cargar train_2016_2017

In [ ]:
DTYPES = {
    "id": "int64",
    "store_nbr": "int16",
    "item_nbr": "int32",
    "unit_sales": "float32",
}

df = pd.read_csv(os.path.join(BASE_DIR, "train_2016_2017.csv"), dtype=DTYPES, parse_dates=["date"])

print("Filas:", f"{len(df):,}")
print("Columnas:", list(df.columns))


## 2. Unir con stores.csv (por store_nbr)
Una fila por tienda, no duplica filas. Renombramos `type` a `store_type` para que no choque más adelante con la columna `type` de `holidays_events.csv`.

In [ ]:
stores = pd.read_csv(os.path.join(BASE_DIR, "stores.csv"))
stores = stores.rename(columns={"type": "store_type"})

df = df.merge(stores, on="store_nbr", how="left")

print("Filas:", f"{len(df):,}", "(debería seguir igual)")
print("Columnas:", list(df.columns))


## 3. Unir con transactions.csv (por date + store_nbr)
Revisé el archivo: no tiene combinaciones repetidas de fecha+tienda, así que tampoco duplica filas.

In [ ]:
transactions = pd.read_csv(os.path.join(BASE_DIR, "transactions.csv"))
transactions["date"] = pd.to_datetime(transactions["date"])

df = df.merge(transactions, on=["date", "store_nbr"], how="left")

print("Filas:", f"{len(df):,}", "(debería seguir igual)")
print("Columnas:", list(df.columns))


## 4. Unir con items.csv (por item_nbr)
Una fila por producto, no duplica filas.

In [ ]:
items = pd.read_csv(os.path.join(BASE_DIR, "items.csv"))

df = df.merge(items, on="item_nbr", how="left")

print("Filas:", f"{len(df):,}", "(debería seguir igual)")
print("Columnas:", list(df.columns))


## 5. Unir con holidays_events.csv (por date)
Acá sí hay que deduplicar: 31 fechas tienen más de una fila (ej. feriado nacional + local el mismo día). Nos quedamos con la primera fila por fecha. También renombramos `type` a `holiday_type`.

In [ ]:
holidays = pd.read_csv(os.path.join(BASE_DIR, "holidays_events.csv"))
holidays["date"] = pd.to_datetime(holidays["date"])

print("Filas antes de deduplicar:", len(holidays))
holidays = holidays.drop_duplicates(subset="date", keep="first")
print("Filas después de deduplicar:", len(holidays))

holidays = holidays.rename(columns={"type": "holiday_type"})


In [ ]:
df = df.merge(holidays, on="date", how="left")

print("Filas:", f"{len(df):,}", "(debería seguir igual)")
print("Columnas:", list(df.columns))


## 6. Unir con oil.csv (por date)
Una fila por fecha, no duplica filas.

In [ ]:
oil = pd.read_csv(os.path.join(BASE_DIR, "oil.csv"))
oil["date"] = pd.to_datetime(oil["date"])

df = df.merge(oil, on="date", how="left")

print("Filas:", f"{len(df):,}", "(debería seguir igual)")
print("Columnas:", list(df.columns))


## 7. Revisión de la base final
Es esperable ver `NaN` en `dcoilwtico` (fines de semana/feriados sin precio de petróleo) y en las columnas de `holidays_events` (días que no son feriado) — no es un error.

In [ ]:
print(f"Filas: {len(df):,}")
print(f"Columnas: {df.shape[1]}")
print()
df.info()


In [ ]:
df.isna().sum()


In [ ]:
df.head(10)


## 8. Guardar el resultado

In [ ]:
OUTPUT_PATH = os.path.join(BASE_DIR, "train_2016_2017_union.csv")
df.to_csv(OUTPUT_PATH, index=False)
print("Guardado en:", OUTPUT_PATH)
